
# Comparison Notebook — Deterministic NN vs Bayesian NN

This notebook compares:

- **Deterministic baseline** trained on the same input features
- **Bayesian Neural Network (BNN)** loaded from the best checkpoint

Both models use the same feature pipeline:

\[
x = [	ext{BERT embeddings}, 	heta_{	ext{LDA}}]
\]

The goal is to check whether the BNN is really adding value beyond a standard neural network.


In [10]:

import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pyro
from pyro.infer import Predictive
import pandas as pd

# Project root setup
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

root_path = os.getcwd()
if root_path not in sys.path:
    sys.path.append(root_path)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from src.data.loader import make_dataloaders, load_tfidf_splits
from src.models.lda import TopicModeler
from src.models.bnn import BayesianClassifier
from src.inference.variational import VariationalInference
from src.models.deterministic import (
    DeterministicClassifier,
    train_one_epoch,
    evaluate_deterministic,
)

pyro.clear_param_store()

DEVICE = "cpu"
print("Working directory:", os.getcwd())
print("Device:", DEVICE)


Working directory: /Users/jimenamonteagudoruiz/Bayesian-Sentiment-Analysis
Device: cpu


## 1. Load LDA and build the same BERT + LDA features used by the BNN

In [11]:

lda_modeler = TopicModeler.load("experiments/results/lda_model.pkl")

X_tr, X_val, X_te, _ = load_tfidf_splits()
theta_tr = lda_modeler.get_topics(X_tr)
theta_val = lda_modeler.get_topics(X_val)
theta_te = lda_modeler.get_topics(X_te)

# IMPORTANT:
# This matches the current project setup. If your loader uses global indices over the full dataset,
# replace this with a full-size theta matrix aligned with the original sample indices.
all_theta = np.vstack([theta_tr, theta_val, theta_te])

train_loader, val_loader, test_loader = make_dataloaders(
    batch_size=64,
    topic_vecs=all_theta
)

x_batch, y_batch = next(iter(train_loader))
print("Train batch X shape:", x_batch.shape)
print("Train batch y shape:", y_batch.shape)
print("Input dimension:", x_batch.shape[1])


Train batch X shape: torch.Size([64, 778])
Train batch y shape: torch.Size([64])
Input dimension: 778


## 2. Train the deterministic baseline

In [12]:

input_dim = x_batch.shape[1]
hidden_dim = 128
lr = 1e-3
epochs = 20

det_model = DeterministicClassifier(input_dim=input_dim, hidden_dim=hidden_dim).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(det_model.parameters(), lr=lr)

best_val_nll_det = float("inf")
best_epoch_det = -1
best_det_path = "experiments/results/deterministic_best.pt"

train_losses_det = []
val_accuracies_det = []
val_nlls_det = []

os.makedirs("experiments/results", exist_ok=True)


In [13]:

for epoch in range(epochs):
    train_loss = train_one_epoch(
        model=det_model,
        data_loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
    )
    train_losses_det.append(train_loss)

    val_metrics_det = evaluate_deterministic(
        model=det_model,
        data_loader=val_loader,
        device=DEVICE,
    )

    val_acc_det = val_metrics_det["accuracy"]
    val_nll_det = val_metrics_det["nll"]

    val_accuracies_det.append(val_acc_det)
    val_nlls_det.append(val_nll_det)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train CE: {train_loss:.4f} | "
        f"Val Acc: {val_acc_det:.4f} | "
        f"Val NLL: {val_nll_det:.4f}"
    )

    if val_nll_det < best_val_nll_det:
        best_val_nll_det = val_nll_det
        best_epoch_det = epoch + 1
        torch.save(det_model.state_dict(), best_det_path)
        print(f"New best deterministic model saved at epoch {best_epoch_det}")

print("\nDeterministic training finished.")
print(f"Best deterministic epoch: {best_epoch_det}")
print(f"Best deterministic validation NLL: {best_val_nll_det:.4f}")
print(f"Best deterministic model saved to: {best_det_path}")


Epoch 1/20 | Train CE: 0.4943 | Val Acc: 0.8032 | Val NLL: 0.4009
New best deterministic model saved at epoch 1
Epoch 2/20 | Train CE: 0.3864 | Val Acc: 0.8302 | Val NLL: 0.3843
New best deterministic model saved at epoch 2
Epoch 3/20 | Train CE: 0.3650 | Val Acc: 0.8302 | Val NLL: 0.3672
New best deterministic model saved at epoch 3
Epoch 4/20 | Train CE: 0.3596 | Val Acc: 0.8352 | Val NLL: 0.3698
Epoch 5/20 | Train CE: 0.3593 | Val Acc: 0.8332 | Val NLL: 0.3656
New best deterministic model saved at epoch 5
Epoch 6/20 | Train CE: 0.3483 | Val Acc: 0.8222 | Val NLL: 0.3908
Epoch 7/20 | Train CE: 0.3438 | Val Acc: 0.8282 | Val NLL: 0.3720
Epoch 8/20 | Train CE: 0.3411 | Val Acc: 0.8352 | Val NLL: 0.3566
New best deterministic model saved at epoch 8
Epoch 9/20 | Train CE: 0.3361 | Val Acc: 0.8422 | Val NLL: 0.3546
New best deterministic model saved at epoch 9
Epoch 10/20 | Train CE: 0.3388 | Val Acc: 0.8442 | Val NLL: 0.3540
New best deterministic model saved at epoch 10
Epoch 11/20 | Tr

## 3. Load the best BNN checkpoint

In [14]:
def evaluate_bnn_metrics(model, guide, data_loader, num_mc_samples=50, device="cpu"):
    model.eval()

    all_probs = []
    all_targets = []

    predictive = Predictive(
        model,
        guide=guide,
        num_samples=num_mc_samples,
        return_sites=["_RETURN"]
    )

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            sampled_logits = predictive(x_batch)["_RETURN"]   # shape: [S, B, 2]
            sampled_probs = torch.softmax(sampled_logits, dim=-1)  # [S, B, 2]
            mean_probs = sampled_probs.mean(dim=0)  # [B, 2]

            all_probs.append(mean_probs.cpu())
            all_targets.append(y_batch.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    preds = all_probs.argmax(dim=1)
    accuracy = (preds == all_targets).float().mean().item()

    eps = 1e-8
    true_class_probs = all_probs[torch.arange(len(all_targets)), all_targets]
    nll = -torch.log(true_class_probs.clamp(min=eps)).mean().item()

    return {
        "accuracy": accuracy,
        "nll": nll,
        "probs": all_probs,
        "targets": all_targets,
    }

pyro.clear_param_store()

bnn_model = BayesianClassifier(input_dim=input_dim, hidden_dim=hidden_dim).to(DEVICE)
vi_engine = VariationalInference(bnn_model, lr=1e-3)

bnn_best_path = "experiments/results/bnn_params_best.pt"
assert os.path.exists(bnn_best_path), f"BNN checkpoint not found: {bnn_best_path}"

pyro.get_param_store().load(bnn_best_path)
print("Loaded BNN checkpoint:", bnn_best_path)

Loaded BNN checkpoint: experiments/results/bnn_params_best.pt


## 4. Compare both models on validation

In [15]:

# Load best deterministic checkpoint
det_model.load_state_dict(torch.load(best_det_path, map_location=DEVICE))

val_metrics_det = evaluate_deterministic(det_model, val_loader, device=DEVICE)
val_metrics_bnn = evaluate_bnn_metrics(
    model=vi_engine.model,
    guide=vi_engine.guide,
    data_loader=val_loader,
    num_mc_samples=50,
    device=DEVICE,
)

comparison_val = pd.DataFrame([
    {
        "Model": "Deterministic",
        "Val Accuracy": val_metrics_det["accuracy"],
        "Val NLL": val_metrics_det["nll"],
    },
    {
        "Model": "BNN",
        "Val Accuracy": val_metrics_bnn["accuracy"],
        "Val NLL": val_metrics_bnn["nll"],
    },
])

comparison_val


RuntimeError: shape '[128, 778]' is invalid for input of size 98690
                  Trace Shapes:            
                   Param Sites:            
         AutoDiagonalNormal.loc 98690      
       AutoDiagonalNormal.scale 98690      
                  Sample Sites:            
_AutoDiagonalNormal_latent dist     | 98690
                          value     | 98690

In [ ]:
x_batch, y_batch = next(iter(val_loader))
logits = vi_engine.model(x_batch)
print(logits.shape)

torch.Size([64, 2])


## 5. Compare both models on test

In [ ]:

test_metrics_det = evaluate_deterministic(det_model, test_loader, device=DEVICE)
test_metrics_bnn = evaluate_bnn_metrics(
    model=vi_engine.model,
    guide=vi_engine.guide,
    data_loader=test_loader,
    num_mc_samples=50,
    device=DEVICE,
)

comparison_test = pd.DataFrame([
    {
        "Model": "Deterministic",
        "Test Accuracy": test_metrics_det["accuracy"],
        "Test NLL": test_metrics_det["nll"],
    },
    {
        "Model": "BNN",
        "Test Accuracy": test_metrics_bnn["accuracy"],
        "Test NLL": test_metrics_bnn["nll"],
    },
])

comparison_test


,Model,Test Accuracy,Test NLL
0,Deterministic,0.849,0.336706
1,BNN,0.784,0.449753



## 6. How to interpret the comparison

- If **BNN has similar accuracy but lower NLL**, it is adding probabilistic value.
- If **BNN has better accuracy and lower NLL**, it is clearly better.
- If **deterministic beats BNN in both**, then the BNN is not helping in predictive performance, and its value would need to come from uncertainty quality, calibration, or decision-with-cost analysis.
